# Distribution of Returns — Full Pipeline (single asset)

Walk through the entire pipeline for a single asset (default: AAPL) at all four
timeframes. Each step shows intermediate results so you can cross-check against
the Excel template cell-by-cell.

The pipeline:
1. Fetch + clean daily / weekly / monthly OHLC from yfinance
2. Compute descriptive stats, bins, and positive-return analysis per return column
3. Derive quarterly from monthly (anchor on latest closed month, group every 3)
4. Preview the row this asset contributes to the Summary Table

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

from dor import clean_data, descriptive_stats, fetch_prices
from dor_bins import compute_bins, positive_return_stats, scheme_for
from dor_quarterly import monthly_to_quarterly

ticker = 'AAPL'
print(f'Asset: {ticker}')

Asset: AAPL


## 1. Daily

Daily has three return columns (C-C, H-L, O-C). Each one runs through the same
descriptive stats / bin / positive-return analysis stack.

In [2]:
daily = clean_data(fetch_prices(ticker, 'd'), 'd')
print(f'{len(daily)} rows from {daily.index[-1]} to {daily.index[0]}')
daily.head()

11440 rows from 12-12-80 to 05-05-26


,Open,High,Low,Close,Adj Close,C-C Returns,H-L Returns,O-C Returns
Date,,,,,,,,
05-05-26,276.929993,284.570007,276.500000,284.179993,284.179993,0.026551,0.029186,0.026180
04-05-26,279.660004,280.630005,274.859985,276.829987,276.829987,-0.011816,0.020993,-0.010119
01-05-26,278.859985,287.220001,278.369995,280.140015,280.140015,0.032394,0.031792,0.004590
30-04-26,270.500000,276.000000,268.140015,271.350006,271.350006,0.004368,0.029313,0.003142
29-04-26,267.549988,271.040009,267.040009,270.170013,270.170013,-0.001995,0.014979,0.009793


In [3]:
def asset_block(frame, tf):
    blocks = {}
    for col in ['C-C Returns', 'H-L Returns', 'O-C Returns']:
        if col not in frame.columns:
            continue
        blocks[col] = {
            'stats': descriptive_stats(frame[col]),
            'positives': positive_return_stats(frame[col]),
            'bins': compute_bins(frame[col], scheme_for(col, tf)),
        }
    return blocks

daily_block = asset_block(daily, 'd')
pd.DataFrame({col: daily_block[col]['stats'] for col in daily_block}).T

,mean,standard_error,median,mode,standard_deviation,sample_variance,kurtosis,skewness,range,minimum,maximum,sum,count
C-C Returns,0.001083,0.000258,0.000000,0.0,0.027575,0.000760,18.694049,-0.362041,0.850972,-0.518692,0.332281,12.393015,11439.0
H-L Returns,0.031262,0.000209,0.025885,0.0,0.022315,0.000498,17.792503,2.702820,0.359158,0.000000,0.359158,357.637358,11440.0
O-C Returns,0.000024,0.000211,0.000000,0.0,0.022588,0.000510,5.799707,0.214181,0.436180,-0.243524,0.192656,0.271207,11440.0


In [4]:
print('=== C-C Returns bins (daily) ===')
display(daily_block['C-C Returns']['bins'])
print('=== H-L Returns bins (daily) ===')
display(daily_block['H-L Returns']['bins'])
print('=== O-C Returns bins (daily) ===')
display(daily_block['O-C Returns']['bins'])

=== C-C Returns bins (daily) ===


,k_label,edge,frequency,probability,cumulative
0,<= -3σ,-0.081640,58,0.005070,0.005070
1,"(-3σ, -2.25σ]",-0.060959,119,0.010403,0.015473
2,"(-2.25σ, -1.5σ]",-0.040278,392,0.034269,0.049742
3,"(-1.5σ, -0.75σ]",-0.019597,1345,0.117580,0.167322
4,"(-0.75σ, 0σ]",0.001083,3991,0.348894,0.516216
5,"(0σ, 0.75σ]",0.021764,3700,0.323455,0.839671
6,"(0.75σ, 1.5σ]",0.042445,1224,0.107002,0.946674
7,"(1.5σ, 2.25σ]",0.063126,392,0.034269,0.980942
8,"(2.25σ, 3σ]",0.083807,128,0.011190,0.992132
9,> 3σ,NaN,90,0.007868,1.000000


=== H-L Returns bins (daily) ===


,k_label,edge,frequency,probability,cumulative
0,<= 0,0.000000,28,0.002448,0.002448
1,"(0σ, 0.4σ]",0.008926,737,0.064423,0.066871
2,"(0.4σ, 0.8σ]",0.017852,2550,0.222902,0.289773
3,"(0.8σ, 1.2σ]",0.026778,2620,0.229021,0.518794
4,"(1.2σ, 1.6σ]",0.035704,1994,0.174301,0.693094
5,"(1.6σ, 2σ]",0.044630,1273,0.111276,0.804371
6,"(2σ, 2.4σ]",0.053557,835,0.072990,0.877360
7,"(2.4σ, 2.8σ]",0.062483,487,0.042570,0.919930
8,"(2.8σ, 3.2σ]",0.071409,325,0.028409,0.948339
9,> 3.2σ,NaN,591,0.051661,1.000000


=== O-C Returns bins (daily) ===


,k_label,edge,frequency,probability,cumulative
0,<= -3σ,-0.067741,69,0.006031,0.006031
1,"(-3σ, -2.25σ]",-0.050800,137,0.011976,0.018007
2,"(-2.25σ, -1.5σ]",-0.033859,411,0.035927,0.053934
3,"(-1.5σ, -0.75σ]",-0.016917,1325,0.115822,0.169755
4,"(-0.75σ, 0σ]",0.000024,4207,0.367745,0.537500
5,"(0σ, 0.75σ]",0.016965,3404,0.297552,0.835052
6,"(0.75σ, 1.5σ]",0.033906,1232,0.107692,0.942745
7,"(1.5σ, 2.25σ]",0.050847,401,0.035052,0.977797
8,"(2.25σ, 3σ]",0.067788,160,0.013986,0.991783
9,> 3σ,NaN,94,0.008217,1.000000


In [5]:
pd.DataFrame({col: daily_block[col]['positives'] for col in daily_block}).T

,average_positive,count_positive,freq_pct_positive,freq_adjusted_return
C-C Returns,0.019951,5717.0,0.499781,0.009971
H-L Returns,0.031339,11412.0,0.997552,0.031262
O-C Returns,0.016859,5291.0,0.462500,0.007797


## 2. Weekly

Weekly drops the O-C column (open and close span many sessions, so the "intraday"
notion no longer applies). Bin schemes are still the daily/weekly variants
(9 edges from -3σ to +3σ in 0.75σ steps for C-C, 0 to 3.2σ in 0.4σ for H-L).

In [6]:
weekly = clean_data(fetch_prices(ticker, 'w'), 'w')
weekly_block = asset_block(weekly, 'w')
pd.DataFrame({col: weekly_block[col]['stats'] for col in weekly_block}).T

,mean,standard_error,median,mode,standard_deviation,sample_variance,kurtosis,skewness,range,minimum,maximum,sum,count
C-C Returns,0.005121,0.001210,0.004171,0.000000,0.058902,0.003469,4.834037,-0.063950,0.903981,-0.506587,0.397394,12.126297,2368.0
H-L Returns,0.083812,0.001204,0.070156,0.119402,0.058585,0.003432,60.006635,4.646178,1.182844,0.004348,1.187191,198.551307,2369.0


In [7]:
display(weekly_block['C-C Returns']['bins'])
display(weekly_block['H-L Returns']['bins'])

,k_label,edge,frequency,probability,cumulative
0,<= -3σ,-0.171586,18,0.007601,0.007601
1,"(-3σ, -2.25σ]",-0.127409,17,0.007179,0.014780
2,"(-2.25σ, -1.5σ]",-0.083233,82,0.034628,0.049409
3,"(-1.5σ, -0.75σ]",-0.039056,322,0.135980,0.185389
4,"(-0.75σ, 0σ]",0.005121,762,0.321791,0.507179
5,"(0σ, 0.75σ]",0.049298,728,0.307432,0.814611
6,"(0.75σ, 1.5σ]",0.093474,298,0.125845,0.940456
7,"(1.5σ, 2.25σ]",0.137651,98,0.041385,0.981841
8,"(2.25σ, 3σ]",0.181828,28,0.011824,0.993666
9,> 3σ,NaN,15,0.006334,1.000000


,k_label,edge,frequency,probability,cumulative
0,<= 0,0.000000,0,0.000000,0.000000
1,"(0σ, 0.4σ]",0.023434,77,0.032503,0.032503
2,"(0.4σ, 0.8σ]",0.046868,521,0.219924,0.252427
3,"(0.8σ, 1.2σ]",0.070302,590,0.249050,0.501477
4,"(1.2σ, 1.6σ]",0.093736,430,0.181511,0.682989
5,"(1.6σ, 2σ]",0.117170,307,0.129591,0.812579
6,"(2σ, 2.4σ]",0.140603,169,0.071338,0.883917
7,"(2.4σ, 2.8σ]",0.164037,104,0.043900,0.927818
8,"(2.8σ, 3.2σ]",0.187471,61,0.025749,0.953567
9,> 3.2σ,NaN,110,0.046433,1.000000


## 3. Monthly

Monthly uses the *monthly-quarterly* bin schemes — finer near the mean
(C-C: 11 edges in 0.6σ steps; H-L: 11 edges in 0.4σ steps up to 4σ),
giving 12 frequency rows instead of 10.

In [8]:
monthly = clean_data(fetch_prices(ticker, 'm'), 'm')
monthly_block = asset_block(monthly, 'm')
pd.DataFrame({col: monthly_block[col]['stats'] for col in monthly_block}).T

,mean,standard_error,median,mode,standard_deviation,sample_variance,kurtosis,skewness,range,minimum,maximum,sum,count
C-C Returns,0.023443,0.005382,0.024746,NaN,0.119737,0.014337,1.678030,-0.143895,1.031219,-0.577436,0.453783,11.604489,495.0
H-L Returns,0.196942,0.005792,0.164779,0.093024,0.128993,0.016639,29.146554,3.840923,1.489715,0.037378,1.527093,97.683278,496.0


In [9]:
display(monthly_block['C-C Returns']['bins'])
display(monthly_block['H-L Returns']['bins'])

,k_label,edge,frequency,probability,cumulative
0,<= -3σ,-0.335766,1,0.002020,0.002020
1,"(-3σ, -2.4σ]",-0.263924,6,0.012121,0.014141
2,"(-2.4σ, -1.8σ]",-0.192082,10,0.020202,0.034343
3,"(-1.8σ, -1.2σ]",-0.120240,30,0.060606,0.094949
4,"(-1.2σ, -0.6σ]",-0.048399,73,0.147475,0.242424
5,"(-0.6σ, 0σ]",0.023443,126,0.254545,0.496970
6,"(0σ, 0.6σ]",0.095285,124,0.250505,0.747475
7,"(0.6σ, 1.2σ]",0.167127,74,0.149495,0.896970
8,"(1.2σ, 1.8σ]",0.238969,32,0.064646,0.961616
9,"(1.8σ, 2.4σ]",0.310811,14,0.028283,0.989899


,k_label,edge,frequency,probability,cumulative
0,<= 0,0.000000,0,0.000000,0.000000
1,"(0σ, 0.4σ]",0.051597,4,0.008065,0.008065
2,"(0.4σ, 0.8σ]",0.103194,74,0.149194,0.157258
3,"(0.8σ, 1.2σ]",0.154792,140,0.282258,0.439516
4,"(1.2σ, 1.6σ]",0.206389,112,0.225806,0.665323
5,"(1.6σ, 2σ]",0.257986,68,0.137097,0.802419
6,"(2σ, 2.4σ]",0.309583,31,0.062500,0.864919
7,"(2.4σ, 2.8σ]",0.361181,24,0.048387,0.913306
8,"(2.8σ, 3.2σ]",0.412778,18,0.036290,0.949597
9,"(3.2σ, 3.6σ]",0.464375,10,0.020161,0.969758


## 4. Quarterly

Quarterly is **derived from monthly**, not fetched. Rule:
- anchor on the latest closed monthly bar
- group every 3 months going *backward* from that anchor (NOT calendar quarters)
- Open = first month's open, Close/Adj Close = last month's
- High = max of all 3 months, Low = min of all 3 months
- drop the oldest group if it has fewer than 3 months

Below: the newest 3 monthly rows that became the newest quarterly row,
side-by-side with the quarterly aggregation.

In [10]:
print('Newest 3 monthly bars (newest first):')
display(monthly[['Open', 'High', 'Low', 'Close', 'Adj Close']].head(3))

quarterly = monthly_to_quarterly(monthly)
print('First (newest) quarterly bar:')
display(quarterly[['Open', 'High', 'Low', 'Close', 'Adj Close']].head(1))

# Sanity: the quarterly High should equal the max of the 3 monthly Highs.
top3 = monthly.head(3)
assert quarterly['High'].iloc[0] == top3['High'].max()
assert quarterly['Low'].iloc[0] == top3['Low'].min()
assert quarterly['Open'].iloc[0] == top3['Open'].iloc[2]   # oldest of the 3
assert quarterly['Close'].iloc[0] == top3['Close'].iloc[0]  # newest
print('quarterly aggregation invariants hold')

Newest 3 monthly bars (newest first):


,Open,High,Low,Close,Adj Close
Date,,,,,
01-04-26,254.080002,276.000000,245.699997,271.350006,271.350006
01-03-26,262.410004,266.529999,245.509995,253.789993,253.789993
01-02-26,260.029999,280.910004,255.449997,264.179993,263.933014


First (newest) quarterly bar:


,Open,High,Low,Close,Adj Close
Date,,,,,
01-04-26,260.029999,280.910004,245.509995,271.350006,271.350006


quarterly aggregation invariants hold


In [11]:
quarterly_block = asset_block(quarterly, 'q')
pd.DataFrame({col: quarterly_block[col]['stats'] for col in quarterly_block}).T

,mean,standard_error,median,mode,standard_deviation,sample_variance,kurtosis,skewness,range,minimum,maximum,sum,count
C-C Returns,0.071674,0.015889,0.067299,NaN,0.203477,0.041403,1.452070,0.204910,1.425916,-0.615006,0.810910,11.754575,164.0
H-L Returns,0.385926,0.021095,0.328863,NaN,0.270976,0.073428,30.718867,4.298059,2.570782,0.093503,2.664285,63.677774,165.0


In [12]:
display(quarterly_block['C-C Returns']['bins'])
display(quarterly_block['H-L Returns']['bins'])

,k_label,edge,frequency,probability,cumulative
0,<= -3σ,-0.538756,1,0.006098,0.006098
1,"(-3σ, -2.4σ]",-0.416670,1,0.006098,0.012195
2,"(-2.4σ, -1.8σ]",-0.294584,2,0.012195,0.024390
3,"(-1.8σ, -1.2σ]",-0.172498,13,0.079268,0.103659
4,"(-1.2σ, -0.6σ]",-0.050412,21,0.128049,0.231707
5,"(-0.6σ, 0σ]",0.071674,46,0.280488,0.512195
6,"(0σ, 0.6σ]",0.193760,47,0.286585,0.798780
7,"(0.6σ, 1.2σ]",0.315846,18,0.109756,0.908537
8,"(1.2σ, 1.8σ]",0.437932,6,0.036585,0.945122
9,"(1.8σ, 2.4σ]",0.560018,6,0.036585,0.981707


,k_label,edge,frequency,probability,cumulative
0,<= 0,0.000000,0,0.000000,0.000000
1,"(0σ, 0.4σ]",0.108390,2,0.012121,0.012121
2,"(0.4σ, 0.8σ]",0.216781,37,0.224242,0.236364
3,"(0.8σ, 1.2σ]",0.325171,41,0.248485,0.484848
4,"(1.2σ, 1.6σ]",0.433562,37,0.224242,0.709091
5,"(1.6σ, 2σ]",0.541952,19,0.115152,0.824242
6,"(2σ, 2.4σ]",0.650343,14,0.084848,0.909091
7,"(2.4σ, 2.8σ]",0.758733,5,0.030303,0.939394
8,"(2.8σ, 3.2σ]",0.867124,4,0.024242,0.963636
9,"(3.2σ, 3.6σ]",0.975514,2,0.012121,0.975758


## 5. Summary row preview

The asset's 8 cells in the final Summary Table: C-C StdDev across D/W/M/Q,
then H-L Return Avg across D/W/M/Q. Paste these into the Excel template's
matching row and the numbers should agree (decimals, format as %).

In [13]:
row = {}
for tf, block in [('Daily', daily_block), ('Weekly', weekly_block), ('Monthly', monthly_block), ('Quarterly', quarterly_block)]:
    row[f'C-C StdDev {tf}'] = block['C-C Returns']['stats']['standard_deviation']
for tf, block in [('Daily', daily_block), ('Weekly', weekly_block), ('Monthly', monthly_block), ('Quarterly', quarterly_block)]:
    row[f'H-L Mean {tf}'] = block['H-L Returns']['stats']['mean']

summary_row = pd.Series(row, name=ticker)
summary_row.to_frame().T.style.format('{:.2%}')

,C-C StdDev Daily,C-C StdDev Weekly,C-C StdDev Monthly,C-C StdDev Quarterly,H-L Mean Daily,H-L Mean Weekly,H-L Mean Monthly,H-L Mean Quarterly
AAPL,2.76%,5.89%,11.97%,20.35%,3.13%,8.38%,19.69%,38.59%
